# RAG powered by LangDB
LangDB aims to bridge the gap between data and AI world by running them alongside. You can create an entire LangDB experience just using SQL commands all the way from extraction to creating a chat agent. 

This sample demonstrates the following:

- Extract data from PDF using `load_pdf_text` command.
- Create a RAG function that leverages vector search easily using `Endpoint`
- Create a LangDB `model` that connects to `OpenAI` APIs.

### Extract semantically chunked data from pdfs and insert into tables.

In [1]:
CREATE TABLE pdfs (
  id UUID DEFAULT generateUUIDv4(),
  content `String`, 
  metadata `String`,
   
) 
engine = MergeTree
order by (id, content);

In [13]:
INSERT INTO pdfs(content, metadata)
SELECT content, metadata from 
load_pdf_text ('https://d18rn0p25nwr6d.cloudfront.net/CIK-0000320193/faab4555-c69b-438a-aaf7-e09305f87ca3.pdf');

In [14]:
select count(*) from pdfs

,count()
0,264


In [15]:
select * from pdfs limit 2;

,id,content,metadata
0,8d3381b5-db68-4bfd-8090-db82d63625f4,"SIGNATURE\r\nPursuant to the requirements of the Securities Exchange Act of 1934, the Registrant has duly caused this report to be signed on its behalf by the\r\nundersigned hereunto duly authorized.\r\nDate: May 3, 2024 Apple Inc.\r\nBy: /s/ Katherine Adams\r\nKatherine Adams\r\nSenior Vice President,\r\nGeneral Counsel and Secretary","{""source"":""file:///var/lib/clickhouse/user_files/apple.pdf"",""chunk_id"":0,""page_no"":3}"
1,afe5da32-246c-4844-80e3-0713b9afce6c,"and/or maintain the Enhancements. The Board further acknowledges and agrees that (i) the Enhancements confer substantial benefits upon\r\nApple and its shareholders; and (ii) Apple’s commitment to adopt, implement, and/or maintain the Enhancements will serve Apple and its\r\nshareholders’ best interests and constitutes fair, reasonable and adequate consideration for the release of the Settled Plaintiffs’ Claims.\r\n- 6 -\r\nEX. B-1 - NOTICE OF PENDENCY AND PROPOSED SETTLEMENT OF SHAREHOLDER DERIVATIVE ACTIONS","{""source"":""file:///var/lib/clickhouse/user_files/apple.pdf"",""chunk_id"":2,""page_no"":70}"


### Embeddings
Langdb supports convenient way to generate embeddings on the fly. You can use the built-in `embed()` function for development and testing. 

You can also use `OpenAI` or other providers.

In [16]:
CREATE TABLE pdf_embeddings (
  id UUID,
  embeddings `Array`(`Float32`)
) 
engine = MergeTree
order by id;

Use `embed()` function to generate embeddings for each chunk and store them into `pdf_embeddings` table.

You can also use `SPAWN TASK` feature to run this in the background. 

In [39]:
INSERT INTO pdf_embeddings
select p.id, embed(content) from pdfs as p LEFT JOIN pdf_embeddings as pe on p.id = pe.id
where p.id != pe.id
order by p.id;

### Vector Search

`Endpoints` let you create acecss endpoints that can be used both for API consumption as well as RAG inputs for LLMs.
                                                                                                           
Here we are creating an endpoint named `similar()` to perform a vector search against similar chunks to feed into our LLM model.

In [41]:
CREATE ENDPOINT similar(query String "Query to search similar sections in pdf documents") AS
WITH tbl AS (
  SELECT CAST(embed($query) AS `Array`(`Float32`)) AS query
)
SELECT 
  p.id as id, 
  content, 
  cosineDistance(embeddings, query) AS similarity 
FROM 
  pdf_embeddings AS pe 
JOIN 
  pdfs AS p ON p.id = pe.id
CROSS JOIN 
  tbl 
ORDER BY 
  similarity DESC 
  LIMIT 5


Sample vector search using `similar()` function:

In [43]:
select * from similar('Apple liabilities') limit 2

,id,content,similarity
0,60fc2fbb-723f-49b9-83c2-dbb38b5ab068,"execution the Amended Stipulation, nor any proceedings taken pursuant to or in connection with the Amended Stipulation, and/or approval of the\r\nSettlement (including any arguments proffered in connection therewith):\r\n- 4 -\r\n[PROPOSED] FINAL JUDGMENT AND ORDER APPROVING DERIVATIVE ACTION SETTLEMENT",0.410379
1,c67d0db2-7e87-48d3-82c9-04547d74cbf2,"dismissed with prejudice. The Parties shall bear their own costs and expenses, except as otherwise expressly provided in the Amended\r\nStipulation and this Judgment.\r\n- 3 -\r\n[PROPOSED] FINAL JUDGMENT AND ORDER APPROVING DERIVATIVE ACTION SETTLEMENT",0.397675


In [45]:
select * from similar('PLEASE NOTE: THERE IS NO PROOF OF CLAIM FORM FOR') limit 2

,id,content,similarity
0,d3d08d70-ed03-4bf8-8220-68ad44cecb4e,"On April 23, 2024, the Federal Court preliminarily approved the Settlement, authorized this Notice to be provided to Apple shareholders, and scheduled\r\nthe Settlement Fairness Hearing to consider whether to grant final approval of the Settlement.\r\nWHAT ARE THE TERMS OF THE SETTLEMENT?\r\nThe full terms and conditions of the Settlement are embodied in the Stipulation, which is on file with the Federal Court. The following is only a\r\nsummary of the Stipulation.\r\nIn consideration of the full settlement and release of the Settled Plaintiffs’ Claims (defined below) against the Released Defendants’ Parties (defined\r\nbelow) and the dismissal with prejudice of the Actions, Defendants and Apple have agreed that Apple shall adopt and implement the corporate governance\r\nenhancements identified in Exhibit A to the Stipulation in the time and manner specified therein.",0.284785
1,afe5da32-246c-4844-80e3-0713b9afce6c,"and/or maintain the Enhancements. The Board further acknowledges and agrees that (i) the Enhancements confer substantial benefits upon\r\nApple and its shareholders; and (ii) Apple’s commitment to adopt, implement, and/or maintain the Enhancements will serve Apple and its\r\nshareholders’ best interests and constitutes fair, reasonable and adequate consideration for the release of the Settled Plaintiffs’ Claims.\r\n- 6 -\r\nEX. B-1 - NOTICE OF PENDENCY AND PROPOSED SETTLEMENT OF SHAREHOLDER DERIVATIVE ACTIONS",0.283581


### Model and Prompt Creation

You can dynamically create models and prompts on the fly using `CREATE` commands that can leverage tools that you have created.

- Here we are attaching `similar` function as a tool.
- You could also nest queries simply using SQL to hard wire the function chaining. 


In [49]:

CREATE MODEL legal_assist(
  input
) ENGINE = OpenAI(api_key = 'sk-proj-xxx', model_name = 'gpt-3.5-turbo')
PROMPT (system "Use the tool 'similar' to query for similar documents based on user query to assist with quickly searching legal documents.",
  human "{{input}}")
TOOLS (similar)
SETTINGS retries = 1;

### Model executiion

Using the created model, we can perform `legal_assist()` query that leverages `similar()` tool and responds. 

This can immediately be plugged into the frontends to also provide a chat interface.

In [51]:
select legal_assist('PLEASE NOTE: THERE IS NO PROOF OF CLAIM FORM FOR')

,legal_assist('PLEASE NOTE: THERE IS NO PROOF OF CLAIM FORM FOR')
0,"I found some sections in legal documents related to your query ""NO PROOF OF CLAIM FORM"":\n\n1. Section about the adoption and maintenance of corporate governance enhancements in relation to shareholder derivative actions settlement.\n2. Summary of the terms of the settlement agreement between Settled Plaintiffs and Released Defendants' Parties.\n3. Corporate governance enhancements identified in the Stipulation of the settlement.\n4. Signature and authorization for a report under the Securities Exchange Act of 1934.\n\nIf you need more information or details on any specific section, please let me know!"
